In [0]:
# Loading the dataset and displaying the top 10 rows
df_products = spark.table("project.bronze.bronze_products")

display(df_products.head(10))

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,ingestion_timestamp
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14,2026-03-10T10:49:10.211Z
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20,2026-03-10T10:49:10.211Z
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15,2026-03-10T10:49:10.211Z
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26,2026-03-10T10:49:10.211Z
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13,2026-03-10T10:49:10.211Z
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200,38,5,11,2026-03-10T10:49:10.211Z
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350,70,24,44,2026-03-10T10:49:10.211Z
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900,40,8,40,2026-03-10T10:49:10.211Z
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400,27,13,17,2026-03-10T10:49:10.211Z
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600,17,10,12,2026-03-10T10:49:10.211Z


In [0]:
df_products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



## Creating Useful Features:

In [0]:
from pyspark.sql.functions import col, when

# Set volume threshold for bulky items 
BULKY_VOLUME_THRESHOLD = 50000

# Add product_volume_cm3 feature
df_products_final = df_products.withColumn(
    "product_volume_cm3",
    col("product_length_cm") * col("product_height_cm") * col("product_width_cm")
)
# Insight: Large items (furniture, appliances) don't go through standard mail; they require "heavy-lift" logistics partners who have different delivery schedules and higher variance.

# Add product_density feature
df_products_final = df_products_final.withColumn(
    "product_density",
    col("product_weight_g") / col("product_volume_cm3")
)
# Insight: Highly dense items (like a box of lead weights) or very low-density items (like a huge pillow) create different storage and transit challenges.

# Add is_bulky_item feature
df_products_final = df_products_final.withColumn(
    "is_bulky_item",
    when(
        (col("product_weight_g") > 30000) | (col("product_volume_cm3") > BULKY_VOLUME_THRESHOLD),
        1
    ).otherwise(0)
)
# Insight: Most standard carriers in Brazil have a 30kg limit. Anything above this is "Bulky" and is a massive indicator of potential delay because it requires a specialized truck.


display(df_products_final.head(5))

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,ingestion_timestamp,product_volume_cm3,product_density,is_bulky_item
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14,2026-03-10T10:49:10.211Z,2240,0.10044642857142858,0
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20,2026-03-10T10:49:10.211Z,10800,0.09259259259259259,0
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15,2026-03-10T10:49:10.211Z,2430,0.06337448559670782,0
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26,2026-03-10T10:49:10.211Z,2704,0.1372041420118343,0
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13,2026-03-10T10:49:10.211Z,4420,0.1414027149321267,0


In [0]:
df_products_final.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- product_volume_cm3: integer (nullable = true)
 |-- product_density: double (nullable = true)
 |-- is_bulky_item: integer (nullable = false)



In [0]:
df_products_required = df_products_final.drop(
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
            "product_length_cm","product_height_cm", "product_width_cm",
            "ingestion_timestamp"
        )

display(df_products_required.head(5))

product_id,product_category_name,product_weight_g,product_volume_cm3,product_density,is_bulky_item
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225,2240,0.10044642857142858,0
3aa071139cb16b67ca9e5dea641aaa2f,artes,1000,10800,0.09259259259259259,0
96bd76ec8810374ed1b65e291975717f,esporte_lazer,154,2430,0.06337448559670782,0
cef67bcfe19066a932b7673e239eb23d,bebes,371,2704,0.1372041420118343,0
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,625,4420,0.1414027149321267,0


In [0]:
# Checking the schema for any mismatch
df_products_required.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_volume_cm3: integer (nullable = true)
 |-- product_density: double (nullable = true)
 |-- is_bulky_item: integer (nullable = false)



In [0]:
# calculating the null values
{col: df_products_required.filter(df_products_required[col].isNull()).count() for col in df_products_required.columns}

{'product_id': 0,
 'product_category_name': 610,
 'product_weight_g': 2,
 'product_volume_cm3': 2,
 'product_density': 2,
 'is_bulky_item': 0}

### Observation: `product_category_name` is in portugese and has 610 missing values

Translate them to english first and fill the missing values with category `unknown`

In [0]:
df_translation = spark.table("project.bronze.bronze_product_category_translation").drop("ingestion_timestamp")
display(df_translation.limit(5))

product_category_name,product_category_name_english
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto
cama_mesa_banho,bed_bath_table
moveis_decoracao,furniture_decor


In [0]:
df_products_with_english = df_products_required.join(
    df_translation,
    on="product_category_name",
    how="left"
)


display(df_products_with_english.head(5))

product_category_name,product_id,product_weight_g,product_volume_cm3,product_density,is_bulky_item,product_category_name_english
perfumaria,1e9e8ef04dbcff4541ed26657ea517e5,225,2240,0.10044642857142858,0,perfumery
artes,3aa071139cb16b67ca9e5dea641aaa2f,1000,10800,0.09259259259259259,0,art
esporte_lazer,96bd76ec8810374ed1b65e291975717f,154,2430,0.06337448559670782,0,sports_leisure
bebes,cef67bcfe19066a932b7673e239eb23d,371,2704,0.1372041420118343,0,baby
utilidades_domesticas,9dc1a7de274444849c219cff195d0b71,625,4420,0.1414027149321267,0,housewares


### Fill the Null Values with category `unknown`

In [0]:
from pyspark.sql.functions import col, when

df_products_new = df_products_with_english.withColumn(
    "product_category_name_english",
    when(
        col("product_category_name_english").isNull(),
        "unknown category"
    ).otherwise(col("product_category_name_english"))
)

# calculating the null values
{col: df_products_new.filter(df_products_new[col].isNull()).count() for col in df_products_new.columns}

{'product_category_name': 610,
 'product_id': 0,
 'product_weight_g': 2,
 'product_volume_cm3': 2,
 'product_density': 2,
 'is_bulky_item': 0,
 'product_category_name_english': 0}

### Clean the category names and drop the duplicates

In [0]:
from pyspark.sql.functions import col, lower, trim

# Clean the English category name (lowercase and trim)
df_products_new = df_products_new.withColumn(
    "product_category_name_english",
    lower(trim(col("product_category_name_english")))
)

# Drop duplicate products by product_id
df_products_new = df_products_new.dropDuplicates(["product_id"])


In [0]:
df_products_new.show(5)

+---------------------+--------------------+----------------+------------------+-------------------+-------------+-----------------------------+
|product_category_name|          product_id|product_weight_g|product_volume_cm3|    product_density|is_bulky_item|product_category_name_english|
+---------------------+--------------------+----------------+------------------+-------------------+-------------+-----------------------------+
|   ferramentas_jardim|e1d1d22e9f8122a4e...|            2150|             19040|0.11292016806722689|            0|                 garden_tools|
|        esporte_lazer|ce5b91848b91118da...|            1388|              9486|0.14632089394897743|            0|               sports_leisure|
|           brinquedos|1c6fb703c624b381a...|             725|              5610|0.12923351158645277|            0|                         toys|
|            papelaria|4e04ffb7dd3739ecf...|             500|              2688|0.18601190476190477|            0|                

In [0]:
# Drop the original category name column
df_products_final = df_products_new.drop("product_category_name")

display(df_products_final.head(4))

product_id,product_weight_g,product_volume_cm3,product_density,is_bulky_item,product_category_name_english
e1d1d22e9f8122a4ec1533b032c12562,2150,19040,0.11292016806722689,0,garden_tools
ce5b91848b91118daffb3af53b747475,1388,9486,0.14632089394897743,0,sports_leisure
1c6fb703c624b381a20f21f757694866,725,5610,0.12923351158645277,0,toys
4e04ffb7dd3739ecfc37de8927dd586c,500,2688,0.18601190476190477,0,stationery


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

silver_products_schema = StructType([
    StructField("product_id", StringType(), False),                         # NOT NULL - primary key
    StructField("product_category_name_english", StringType(), False),      # NOT NULL - cleaned/translated category
    StructField("product_weight_g", DoubleType(), True),                    # Can be NULL
    StructField("product_volume_cm3", DoubleType(), True),                  # New feature: volume
    StructField("product_density", DoubleType(), True),                     # New feature: density
    StructField("is_bulky_item", IntegerType(), True),                      # New feature: bulky flag
    StructField("ingestion_timestamp", TimestampType(), False)              # NOT NULL - metadata
])

In [0]:
# saving the cleaned data
df_products_final.write.format('delta').mode('overwrite')\
    .option("mergeSchema", "false") \
    .option("schema",silver_products_schema.json())\
    .saveAsTable("project.silver.silver_product")